# GuideSep inference

Place `wav/guide.wav` and `wav/mix.wav`, choose `MODE`, then run all cells.

- `MODE = "waveform"` (default): guide waveform conditioning; positive/negative mel-mask channels disabled.
- `MODE = "pseudo-mask"`: the same guide waveform plus GuideSep's upstream pseudo masks derived from the guide and mixture mel spectrograms, passed through the checkpoint's learned mask MLP.

The pseudo masks are conditioning only; they are not multiplied onto the output spectrum. Final WAV outputs are resampled back to the original `wav/mix.wav` sample rate.


In [ ]:
from pathlib import Path
import sys
import urllib.request

UPSTREAM_COMMIT = "f6bcbdee55b56909a6dc4096f8f75fb93b47c202"
RUNTIME_ROOT = Path(".runtime")
UPSTREAM_FILES = [
    "src/models/backbones/utils.py",
    "src/models/backbones/operator_utils.py",
    "src/models/backbones/layer_utils.py",
    "src/models/backbones/attention_utils.py",
    "src/models/backbones/conditioner.py",
    "src/models/backbones/unet2d.py",
    "src/models/components/utils.py",
    "src/models/components/diffusion.py",
    "src/models/components/sampler.py",
    "src/models/components/scheduler.py",
]

for package in ["src", "src/models", "src/models/backbones", "src/models/components"]:
    package_dir = RUNTIME_ROOT / package
    package_dir.mkdir(parents=True, exist_ok=True)
    (package_dir / "__init__.py").touch()

for relative_path in UPSTREAM_FILES:
    destination = RUNTIME_ROOT / relative_path
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://raw.githubusercontent.com/YutongWen/GuideSep/{UPSTREAM_COMMIT}/{relative_path}"
        urllib.request.urlretrieve(url, destination)

runtime_path = str(RUNTIME_ROOT.resolve())
if runtime_path not in sys.path:
    sys.path.insert(0, runtime_path)

print(f"GuideSep runtime source ready: {RUNTIME_ROOT.resolve()}")


In [ ]:
import math

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
from scipy.ndimage import gaussian_filter
from scipy.signal import resample_poly
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

from src.models.backbones.unet2d import UNet2dBase
from src.models.components.diffusion import EluDiffusion
from src.models.components.sampler import DPMSampler
from src.models.components.scheduler import KarrasSchedule

MODE = "waveform"  # "waveform" or "pseudo-mask"

SAMPLE_RATE = 16000
N_FFT = 510
HOP_LENGTH = 256
N_MELS = 80
SPEC_ABS_EXPONENT = 0.2
SPEC_FACTOR = 0.6
SAMPLER_STEPS = 8
SEED = 42

VALID_MODES = {"waveform", "pseudo-mask"}
if MODE not in VALID_MODES:
    raise ValueError(f"MODE must be one of {sorted(VALID_MODES)}, got {MODE!r}")

MIX_PATH = Path("wav/mix.wav")
GUIDE_PATH = Path("wav/guide.wav")
OUTPUTS = {
    "waveform": (
        Path("output/target_guidesep_waveform.wav"),
        Path("output/residual_guidesep_waveform.wav"),
    ),
    "pseudo-mask": (
        Path("output/target_guidesep_pseudo-mask.wav"),
        Path("output/residual_guidesep_pseudo-mask.wav"),
    ),
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def spec_fwd(spec):
    if SPEC_ABS_EXPONENT != 1:
        spec = spec.abs() ** SPEC_ABS_EXPONENT * torch.exp(1j * spec.angle())
    return spec * SPEC_FACTOR


def spec_back(spec):
    spec = spec / SPEC_FACTOR
    if SPEC_ABS_EXPONENT != 1:
        spec = spec.abs() ** (1 / SPEC_ABS_EXPONENT) * torch.exp(1j * spec.angle())
    return spec


def load_audio(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path.resolve()}")

    audio, sr = sf.read(str(path), always_2d=True, dtype="float32")
    audio = audio.mean(axis=1)

    if sr != SAMPLE_RATE:
        gcd = math.gcd(sr, SAMPLE_RATE)
        audio = resample_poly(
            audio,
            SAMPLE_RATE // gcd,
            sr // gcd,
        ).astype(np.float32, copy=False)

    waveform = torch.from_numpy(np.ascontiguousarray(audio)).unsqueeze(0)
    peak = waveform.abs().max()
    if peak > 0:
        waveform = waveform / peak
    return waveform


def resample_output_to_input_rate(audio, input_sr, input_frames):
    audio = np.asarray(audio, dtype=np.float32)
    if input_sr != SAMPLE_RATE:
        gcd = math.gcd(int(SAMPLE_RATE), int(input_sr))
        audio = resample_poly(
            audio,
            int(input_sr) // gcd,
            int(SAMPLE_RATE) // gcd,
        ).astype(np.float32, copy=False)
    if audio.shape[-1] > input_frames:
        audio = audio[:input_frames]
    elif audio.shape[-1] < input_frames:
        audio = np.pad(audio, (0, input_frames - audio.shape[-1]))
    return audio


class SingleLayerMLP(nn.Module):
    def __init__(self, input_size=80, hidden_size=256, output_size=256):
        super().__init__()
        self.hidden_layer = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.transpose(x, -2, -1)
        x = self.activation(self.hidden_layer(x))
        x = self.output_layer(x)
        return torch.transpose(x, -2, -1)


def build_models():
    net = UNet2dBase(
        cond_on_text=True,
        text_embed_dim=768,
        max_text_len=120,
        dim=128,
        cond_drop_prob=0.0,
        dim_mults=[1, 2, 2, 2],
        channels=2,
        condition_channels=5,
        num_resnet_blocks=2,
        resnet_groups=8,
        layer_attns=[False, False, True, True],
        layer_cross_attns=[False, False, True, True],
        attn_heads=2,
        ff_mult=2.0,
        memory_efficient=True,
        use_condition_block=True,
        mixture_input=False,
    )
    mask_mlp = SingleLayerMLP(
        input_size=N_MELS,
        hidden_size=256,
        output_size=N_FFT // 2 + 1,
    )

    ckpt_path = hf_hub_download(repo_id="YutongCooper/GuideSep-v1", filename="last.ckpt")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint.get("state_dict", checkpoint)

    net_state = {key[4:]: value for key, value in state_dict.items() if key.startswith("net.")}
    if not net_state:
        raise RuntimeError("The checkpoint does not contain the expected 'net.' state_dict keys.")
    missing, unexpected = net.load_state_dict(net_state, strict=False)
    missing = [key for key in missing if not key.startswith("text_conditioner.")]
    if missing or unexpected:
        raise RuntimeError(f"Checkpoint/network mismatch. Missing: {missing}; unexpected: {unexpected}")

    if MODE == "pseudo-mask":
        mlp_state = {key[4:]: value for key, value in state_dict.items() if key.startswith("mlp.")}
        if not mlp_state:
            raise RuntimeError("The checkpoint does not contain the learned GuideSep mask MLP ('mlp.' keys).")
        mask_mlp.load_state_dict(mlp_state, strict=True)

    return net.to(DEVICE).eval(), mask_mlp.to(DEVICE).eval()


net, mask_mlp = build_models()
diffusion = EluDiffusion(sigma_data=0.2, dynamic_threshold=0.0).to(DEVICE)
sampler = DPMSampler(
    num_steps=SAMPLER_STEPS,
    cond_scale=1,
    order=1,
    multisteps=True,
    x0_pred=False,
    log_time_spacing=False,
).to(DEVICE)
noise_schedule = KarrasSchedule(
    sigma_min=0.002,
    sigma_max=80.0,
    rho=7.0,
    num_steps=SAMPLER_STEPS,
)().to(DEVICE)
window = torch.hann_window(N_FFT, periodic=True, device=DEVICE)

torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("Mode:", MODE)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Model loaded.")


In [ ]:
def _hz_to_mel(freq):
    return 2595.0 * torch.log10(1.0 + freq / 700.0)


def _mel_to_hz(mels):
    return 700.0 * (torch.pow(10.0, mels / 2595.0) - 1.0)


def build_mel_filterbank(n_freqs, n_mels, sample_rate, device):
    # Reproduces torchaudio.transforms.MelScale defaults (HTK scale, norm=None).
    all_freqs = torch.linspace(0.0, sample_rate / 2.0, n_freqs, device=device)
    m_min = _hz_to_mel(torch.tensor(0.0, device=device))
    m_max = _hz_to_mel(torch.tensor(sample_rate / 2.0, device=device))
    m_pts = torch.linspace(m_min, m_max, n_mels + 2, device=device)
    f_pts = _mel_to_hz(m_pts)
    f_diff = f_pts[1:] - f_pts[:-1]
    slopes = f_pts.unsqueeze(0) - all_freqs.unsqueeze(1)
    down_slopes = (-slopes[:, :-2]) / f_diff[:-1]
    up_slopes = slopes[:, 2:] / f_diff[1:]
    return torch.max(
        torch.zeros(1, dtype=all_freqs.dtype, device=device),
        torch.min(down_slopes, up_slopes),
    )


def wav_to_mel_from_stft(spec_raw, mel_filterbank):
    # Matches upstream wav_to_mel(): magnitude -> MelScale -> global-max normalization.
    magnitude = spec_raw.abs()
    mel_spec = torch.matmul(
        magnitude.transpose(-1, -2),
        mel_filterbank,
    ).transpose(-1, -2)
    max_value = mel_spec.max()
    if max_value > 0:
        mel_spec = mel_spec / max_value
    return mel_spec


def pseudo_mel_mask(mel_spec, threshold=0.1, sigma=5.0):
    # Matches upstream test-time pseudo-mask settings:
    # threshold=0.1, max_mask_drop_prob=0.0, Gaussian sigma=5.
    masks = []
    for item in mel_spec.detach().cpu().numpy():
        mask = (item > threshold).astype(np.float32)
        mask = gaussian_filter(mask, sigma=sigma)
        max_value = float(mask.max())
        if max_value > 0:
            mask = mask / max_value
        masks.append(mask.astype(np.float32, copy=False))
    return torch.from_numpy(np.stack(masks)).to(DEVICE)


In [ ]:
mixture = load_audio(MIX_PATH).to(DEVICE)
guide = load_audio(GUIDE_PATH).to(DEVICE)

common_samples = min(mixture.shape[-1], guide.shape[-1])
mixture = mixture[..., :common_samples]
guide = guide[..., :common_samples]

stft_kwargs = dict(
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    center=True,
    normalized=True,
    return_complex=True,
)
spec_mix_raw = torch.stft(mixture, window=window, **stft_kwargs)
spec_guide_raw = torch.stft(guide, window=window, **stft_kwargs)

time_length = min(spec_mix_raw.shape[-1], spec_guide_raw.shape[-1]) // 16 * 16
if time_length < 16:
    raise ValueError("Input audio is too short for GuideSep inference.")

spec_mix_raw = spec_mix_raw[..., :time_length]
spec_guide_raw = spec_guide_raw[..., :time_length]

spec_mix = spec_fwd(spec_mix_raw).unsqueeze(1)
spec_mix = torch.cat((spec_mix.real, spec_mix.imag), dim=1)
spec_guide = spec_fwd(spec_guide_raw).unsqueeze(1).abs()

if MODE == "waveform":
    mask_channels = torch.full(
        (mixture.shape[0], 2, N_FFT // 2 + 1, time_length),
        -1.0,
        dtype=spec_mix.dtype,
        device=DEVICE,
    )
else:
    mel_filterbank = build_mel_filterbank(
        n_freqs=N_FFT // 2 + 1,
        n_mels=N_MELS,
        sample_rate=SAMPLE_RATE,
        device=DEVICE,
    )
    guide_mel = wav_to_mel_from_stft(spec_guide_raw, mel_filterbank)
    mix_mel = wav_to_mel_from_stft(spec_mix_raw, mel_filterbank)

    pseudo_positive = pseudo_mel_mask(guide_mel, threshold=0.1, sigma=5.0)
    pseudo_negative = pseudo_mel_mask(mix_mel, threshold=0.1, sigma=5.0)
    pseudo_mel = torch.cat(
        (pseudo_positive.unsqueeze(1), pseudo_negative.unsqueeze(1)),
        dim=1,
    )

    with torch.inference_mode():
        mask_channels = mask_mlp(pseudo_mel)

    print(
        "Guide pseudo-mask range:",
        float(pseudo_positive.min()),
        float(pseudo_positive.max()),
    )
    print(
        "Mix pseudo-mask range:",
        float(pseudo_negative.min()),
        float(pseudo_negative.max()),
    )

condition = torch.cat((spec_mix, spec_guide, mask_channels), dim=1)

torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

initial_noise = torch.randn(
    (mixture.shape[0], 2, N_FFT // 2 + 1, time_length),
    device=DEVICE,
)

print(f"Input duration: {common_samples / SAMPLE_RATE:.3f} s")


In [ ]:
with torch.inference_mode():
    predicted = sampler(
        initial_noise,
        fn=diffusion.denoise_fn,
        net=net,
        sigmas=noise_schedule,
        inj_channels=condition,
        mixture=spec_mix,
        mask=None,
        source=None,
    )

predicted = torch.permute(predicted, (0, 2, 3, 1)).contiguous()
predicted_complex = torch.view_as_complex(predicted)[:, None, :, :]
predicted_complex = spec_back(predicted_complex)

target = torch.istft(
    predicted_complex.squeeze(1),
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    center=True,
    normalized=True,
    window=window,
).detach().cpu().squeeze(0)

mixture_reference = mixture.detach().cpu().squeeze(0)[:target.numel()]
residual = mixture_reference - target

input_info = sf.info(str(MIX_PATH))
input_sample_rate = int(input_info.samplerate)
input_frames = int(input_info.frames)
target_output = resample_output_to_input_rate(target.numpy(), input_sample_rate, input_frames)
residual_output = resample_output_to_input_rate(residual.numpy(), input_sample_rate, input_frames)

target_path, residual_path = OUTPUTS[MODE]
target_path.parent.mkdir(parents=True, exist_ok=True)
sf.write(str(target_path), target_output, input_sample_rate, subtype="FLOAT")
sf.write(str(residual_path), residual_output, input_sample_rate, subtype="FLOAT")

reconstruction_error = (
    mixture_reference - (target + residual)
).abs().max().item()

print("Mode:", MODE)
print("Saved target:", target_path.resolve())
print("Saved residual:", residual_path.resolve())
print(f"Inference sample rate: {SAMPLE_RATE} Hz")
print(f"Output sample rate: {input_sample_rate} Hz")
print(f"Output duration: {target_output.shape[-1] / input_sample_rate:.3f} s")
print(f"Reconstruction max error at inference rate: {reconstruction_error:.3e}")

print(f"\n{MODE.upper()} — Target")
display(Audio(target_output, rate=input_sample_rate))
print(f"{MODE.upper()} — Residual")
display(Audio(residual_output, rate=input_sample_rate))
